In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("balanced_1000_per_month.csv")

print("Loaded:", df.shape)
print(df.columns)

Loaded: (47744, 6)
Index(['Date', 'YearMonth', 'EPI_ISL', 'Country', 'Sequence', 'Length'], dtype='object')


In [4]:
# remove bad chars
df["Sequence"] = df["Sequence"].str.replace("X", "-", regex=False)
df["Sequence"] = df["Sequence"].str.replace("*", "-", regex=False)

seqs = df["Sequence"].astype(str).tolist()

max_len = max(len(s) for s in seqs)
seqs = [s.ljust(max_len, "-") for s in seqs]

char_matrix = np.array([list(s) for s in seqs])

print("Matrix shape:", char_matrix.shape)

Matrix shape: (47744, 1274)


In [5]:
from sklearn.preprocessing import LabelEncoder

X = np.zeros(char_matrix.shape, dtype=np.int16)

for col in range(char_matrix.shape[1]):
    le = LabelEncoder()
    X[:, col] = le.fit_transform(char_matrix[:, col])

print("Encoded:", X.shape)

Encoded: (47744, 1274)


In [6]:
from sklearn.decomposition import PCA

pca = PCA(n_components=25, random_state=42)
X_red = pca.fit_transform(X)

print("Reduced:", X_red.shape)
print("Variance:", pca.explained_variance_ratio_.sum())

Reduced: (47744, 25)
Variance: 0.948163259409735


In [7]:
from sklearn.cluster import KMeans, AgglomerativeClustering

k = 8

# KMeans
km = KMeans(n_clusters=k, random_state=42, n_init=10)
df["Cluster_KMeans"] = km.fit_predict(X_red)

# Hierarchical
hier = AgglomerativeClustering(n_clusters=k)
df["Cluster_Hier"] = hier.fit_predict(X_red)

print(df["Cluster_KMeans"].value_counts())
print(df["Cluster_Hier"].value_counts())


Cluster_KMeans
1    14588
5     9657
3     8024
2     6057
7     5237
4     2611
6      967
0      603
Name: count, dtype: int64
Cluster_Hier
0    14978
3     9661
6     8233
1     6080
2     5092
4     2130
5      966
7      604
Name: count, dtype: int64


In [8]:
from sklearn.metrics import pairwise_distances

sample_df = df.sample(5000, random_state=42)
sample_idx = sample_df.index

X_sample = X_red[sample_idx]

dist_matrix = pairwise_distances(X_sample, metric="hamming")

agg_dist = AgglomerativeClustering(
    n_clusters=k,
    metric="precomputed",
    linkage="average"
)

sample_df["Cluster_Dist"] = agg_dist.fit_predict(dist_matrix)

print(sample_df["Cluster_Dist"].value_counts())

Cluster_Dist
0    4853
1     133
2       8
6       2
7       1
5       1
4       1
3       1
Name: count, dtype: int64


In [9]:
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score

print("Silhouette:", silhouette_score(X_red, df["Cluster_KMeans"]))
print("Davies-Bouldin:", davies_bouldin_score(X_red, df["Cluster_KMeans"]))

print("\n=== METHOD AGREEMENT ===")
print("KMeans vs Hier:",
      adjusted_rand_score(df["Cluster_KMeans"], df["Cluster_Hier"]))

print("KMeans vs Distance:",
      adjusted_rand_score(sample_df["Cluster_KMeans"],
                          sample_df["Cluster_Dist"]))

Silhouette: 0.6602369965134454
Davies-Bouldin: 0.8753009911021002

=== METHOD AGREEMENT ===
KMeans vs Hier: 0.9598281461002488
KMeans vs Distance: -0.011024420226867587


In [10]:
from collections import Counter

df["YearMonth"] = pd.to_datetime(df["YearMonth"])

early_df = df[df["YearMonth"] == df["YearMonth"].min()]

def build_consensus(seqs):
    seqs = [s.ljust(max(len(x) for x in seqs), "-") for s in seqs]
    consensus = ""
    for i in range(len(seqs[0])):
        col = [s[i] for s in seqs]
        consensus += Counter(col).most_common(1)[0][0]
    return consensus

ref_seq = build_consensus(early_df["Sequence"].tolist())

print("Reference length:", len(ref_seq))

Reference length: 1274


In [11]:
def mutation_count(seq, ref):
    return sum(1 for a, b in zip(seq, ref)
               if a != b and a not in ["X", "*", "-"])

df["Mutation_Count"] = df["Sequence"].apply(lambda x: mutation_count(x, ref_seq))

print(df.groupby("Cluster_KMeans")["Mutation_Count"].mean())

Cluster_KMeans
0    1169.809287
1       4.330409
2    1035.530130
3    1138.751745
4      37.319801
5    1148.162576
6    1008.822130
7    1091.837311
Name: Mutation_Count, dtype: float64


In [12]:
# ==========================================================
# TRUE EMERGING MUTATIONS (GROWTH-BASED)
# ==========================================================

from collections import defaultdict
import numpy as np

df = df.sort_values("YearMonth")

mutation_time = defaultdict(lambda: defaultdict(int))

# count mutations per month
for _, row in df.iterrows():
    ym = row["YearMonth"]
    seq = row["Sequence"]

    for i, (a, b) in enumerate(zip(seq, ref_seq)):
        if a != b and a not in ["X", "*", "-"]:
            mutation_time[i][ym] += 1

emerging = []

for pos, time_dict in mutation_time.items():
    times = sorted(time_dict.keys())

    if len(times) < 5:
        continue

    values = np.array([time_dict[t] for t in times])

    # normalize (percentage of sequences per month)
    values = values / 1000.0

    # measure growth
    start = np.mean(values[:3])     # early months
    end = np.mean(values[-3:])      # late months

    growth = end - start

    # condition for true emergence
    if growth > 0.3:   # 30% increase
        emerging.append((pos, start, end, growth))

# sort by strongest growth
emerging = sorted(emerging, key=lambda x: -x[3])[:20]

print("\nTRUE EMERGING MUTATIONS (GROWTH-BASED):\n")

for pos, start, end, growth in emerging:
    print(f"Pos {pos} | Start: {start:.2f} → End: {end:.2f} | Growth: {growth:.2f}")


TRUE EMERGING MUTATIONS (GROWTH-BASED):

Pos 654 | Start: 0.02 → End: 0.99 | Growth: 0.98
Pos 338 | Start: 0.02 → End: 0.99 | Growth: 0.97
Pos 345 | Start: 0.02 → End: 0.99 | Growth: 0.97
Pos 483 | Start: 0.02 → End: 0.98 | Growth: 0.97
Pos 489 | Start: 0.02 → End: 0.98 | Growth: 0.97
Pos 476 | Start: 0.02 → End: 0.98 | Growth: 0.97
Pos 500 | Start: 0.02 → End: 0.98 | Growth: 0.97
Pos 763 | Start: 0.02 → End: 0.98 | Growth: 0.97
Pos 477 | Start: 0.02 → End: 0.98 | Growth: 0.97
Pos 497 | Start: 0.02 → End: 0.98 | Growth: 0.97
Pos 444 | Start: 0.01 → End: 0.98 | Growth: 0.97
Pos 459 | Start: 0.01 → End: 0.98 | Growth: 0.97
Pos 968 | Start: 0.02 → End: 0.98 | Growth: 0.96
Pos 795 | Start: 0.02 → End: 0.98 | Growth: 0.96
Pos 367 | Start: 0.01 → End: 0.97 | Growth: 0.96
Pos 455 | Start: 0.01 → End: 0.97 | Growth: 0.96
Pos 439 | Start: 0.01 → End: 0.97 | Growth: 0.96
Pos 372 | Start: 0.01 → End: 0.97 | Growth: 0.96
Pos 416 | Start: 0.01 → End: 0.97 | Growth: 0.96
Pos 375 | Start: 0.02 → End

In [13]:
cluster_month = pd.crosstab(df["YearMonth"], df["Cluster_KMeans"])

print("\nDOMINANT CLUSTER PER MONTH:\n")
print(cluster_month.idxmax(axis=1))


DOMINANT CLUSTER PER MONTH:

YearMonth
2020-01-01    1
2020-02-01    1
2020-03-01    1
2020-04-01    1
2020-05-01    1
2020-06-01    1
2020-07-01    1
2020-08-01    1
2020-09-01    1
2020-10-01    1
2020-11-01    1
2020-12-01    1
2021-01-01    1
2021-02-01    1
2021-03-01    1
2021-04-01    7
2021-05-01    7
2021-06-01    2
2021-07-01    2
2021-08-01    2
2021-09-01    2
2021-10-01    2
2021-11-01    2
2021-12-01    7
2022-01-01    7
2022-02-01    7
2022-03-01    3
2022-04-01    3
2022-05-01    3
2022-06-01    3
2022-07-01    3
2022-08-01    3
2022-09-01    3
2022-10-01    3
2022-11-01    3
2022-12-01    3
2023-01-01    5
2023-02-01    5
2023-03-01    5
2023-04-01    5
2023-05-01    5
2023-06-01    5
2023-07-01    5
2023-08-01    5
2023-09-01    5
2023-10-01    5
2023-11-01    5
2023-12-01    0
dtype: int32


In [14]:
df.to_csv("final_pipeline_output.csv", index=False)
sample_df.to_csv("sample_distance_clusters.csv", index=False)

print("Saved final outputs.")

Saved final outputs.


In [15]:
# ==========================================================
# LINK CLUSTERS TO MUTATION PROFILES
# ==========================================================

from collections import defaultdict

cluster_mutations = defaultdict(lambda: defaultdict(int))

for _, row in df.iterrows():
    cluster = row["Cluster_KMeans"]
    seq = row["Sequence"]

    for i, (a, b) in enumerate(zip(seq, ref_seq)):
        if a != b and a not in ["X", "*", "-"]:
            cluster_mutations[cluster][i] += 1

# get top mutations per cluster
print("\nTOP MUTATIONS PER CLUSTER:\n")

for cluster in sorted(cluster_mutations.keys()):
    muts = cluster_mutations[cluster]
    top = sorted(muts.items(), key=lambda x: -x[1])[:10]

    print(f"\nCluster {cluster}:")
    for pos, count in top:
        print(f"  Pos {pos} | Count {count}")


TOP MUTATIONS PER CLUSTER:


Cluster 0:
  Pos 49 | Count 603
  Pos 70 | Count 603
  Pos 110 | Count 603
  Pos 111 | Count 603
  Pos 112 | Count 603
  Pos 113 | Count 603
  Pos 114 | Count 603
  Pos 115 | Count 603
  Pos 117 | Count 603
  Pos 118 | Count 603

Cluster 1:
  Pos 613 | Count 12589
  Pos 483 | Count 1716
  Pos 654 | Count 1564
  Pos 500 | Count 1557
  Pos 416 | Count 1409
  Pos 1175 | Count 1308
  Pos 680 | Count 1271
  Pos 476 | Count 1005
  Pos 17 | Count 1000
  Pos 221 | Count 962

Cluster 2:
  Pos 609 | Count 6042
  Pos 637 | Count 6042
  Pos 638 | Count 6042
  Pos 640 | Count 6042
  Pos 641 | Count 6042
  Pos 1060 | Count 6042
  Pos 1062 | Count 6042
  Pos 1064 | Count 6042
  Pos 602 | Count 6041
  Pos 603 | Count 6041

Cluster 3:
  Pos 30 | Count 8016
  Pos 32 | Count 8016
  Pos 31 | Count 8015
  Pos 33 | Count 8014
  Pos 1079 | Count 8013
  Pos 1080 | Count 8013
  Pos 1069 | Count 8012
  Pos 1070 | Count 8012
  Pos 1076 | Count 8012
  Pos 1075 | Count 8011

Cluster 4

In [16]:
# ==========================================================
# WAVE-SPECIFIC MUTATIONS (cluster-linked emergence)
# ==========================================================

wave_mutations = []

for pos, time_dict in mutation_time.items():
    times = sorted(time_dict.keys())
    values = np.array([time_dict[t] for t in times]) / 1000.0

    if len(values) < 6:
        continue

    peak = np.max(values)
    peak_idx = np.argmax(values)

    # check if it rises and then drops (wave pattern)
    if peak > 0.4:
        before = np.mean(values[:peak_idx]) if peak_idx > 2 else 0
        after = np.mean(values[peak_idx+2:]) if peak_idx < len(values)-3 else 0

        if before < 0.2 and after < 0.5:
            wave_mutations.append((pos, peak, times[peak_idx]))

# sort
wave_mutations = sorted(wave_mutations, key=lambda x: -x[1])[:20]

print("\nWAVE-SPECIFIC MUTATIONS:\n")

for pos, peak, t in wave_mutations:
    print(f"Pos {pos} | Peak: {peak:.2f} at {t.date()}")


WAVE-SPECIFIC MUTATIONS:

Pos 150 | Peak: 0.99 at 2022-03-01
Pos 230 | Peak: 0.99 at 2022-03-01
Pos 71 | Peak: 0.94 at 2022-01-01
Pos 72 | Peak: 0.94 at 2022-01-01
Pos 81 | Peak: 0.94 at 2022-01-01
Pos 126 | Peak: 0.94 at 2022-01-01
Pos 121 | Peak: 0.94 at 2022-01-01
Pos 451 | Peak: 0.92 at 2021-08-01
Pos 448 | Peak: 0.91 at 2022-01-01
Pos 1246 | Peak: 0.88 at 2021-11-01
Pos 225 | Peak: 0.87 at 2021-11-01
Pos 860 | Peak: 0.87 at 2021-11-01
Pos 1021 | Peak: 0.87 at 2021-11-01
Pos 1249 | Peak: 0.87 at 2021-11-01
Pos 681 | Peak: 0.87 at 2021-11-01
Pos 1270 | Peak: 0.87 at 2021-11-01
Pos 240 | Peak: 0.87 at 2021-11-01
Pos 373 | Peak: 0.87 at 2021-11-01
Pos 386 | Peak: 0.87 at 2021-11-01
Pos 412 | Peak: 0.87 at 2021-11-01


In [19]:
# ==========================================================
# BUILD DOMINANT CLUSTER TIMELINE
# ==========================================================

# make sure Cluster_KMeans exists
cluster_month = pd.crosstab(df["YearMonth"], df["Cluster_KMeans"])

# dominant cluster per month
dominant = cluster_month.idxmax(axis=1)

print("\nDOMINANT CLUSTER PER MONTH:\n")
print(dominant.head())


DOMINANT CLUSTER PER MONTH:

YearMonth
2020-01-01    1
2020-02-01    1
2020-03-01    1
2020-04-01    1
2020-05-01    1
dtype: int32


In [20]:
print("\nMUTATION → WAVE → CLUSTER\n")

for pos, peak, peak_time in wave_mutations:
    
    if peak_time in dominant.index:
        cluster = dominant.loc[peak_time]
    else:
        continue

    print(f"Pos {pos} | Peak: {peak:.2f} | Month: {peak_time.date()} | Cluster: {cluster}")


MUTATION → WAVE → CLUSTER

Pos 150 | Peak: 0.99 | Month: 2022-03-01 | Cluster: 3
Pos 230 | Peak: 0.99 | Month: 2022-03-01 | Cluster: 3
Pos 71 | Peak: 0.94 | Month: 2022-01-01 | Cluster: 7
Pos 72 | Peak: 0.94 | Month: 2022-01-01 | Cluster: 7
Pos 81 | Peak: 0.94 | Month: 2022-01-01 | Cluster: 7
Pos 126 | Peak: 0.94 | Month: 2022-01-01 | Cluster: 7
Pos 121 | Peak: 0.94 | Month: 2022-01-01 | Cluster: 7
Pos 451 | Peak: 0.92 | Month: 2021-08-01 | Cluster: 2
Pos 448 | Peak: 0.91 | Month: 2022-01-01 | Cluster: 7
Pos 1246 | Peak: 0.88 | Month: 2021-11-01 | Cluster: 2
Pos 225 | Peak: 0.87 | Month: 2021-11-01 | Cluster: 2
Pos 860 | Peak: 0.87 | Month: 2021-11-01 | Cluster: 2
Pos 1021 | Peak: 0.87 | Month: 2021-11-01 | Cluster: 2
Pos 1249 | Peak: 0.87 | Month: 2021-11-01 | Cluster: 2
Pos 681 | Peak: 0.87 | Month: 2021-11-01 | Cluster: 2
Pos 1270 | Peak: 0.87 | Month: 2021-11-01 | Cluster: 2
Pos 240 | Peak: 0.87 | Month: 2021-11-01 | Cluster: 2
Pos 373 | Peak: 0.87 | Month: 2021-11-01 | Cluster: 2